In [14]:
%pip install -q deep_translator 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
# Install deep_translator if not already installed
import subprocess
import sys

try:
    import deep_translator
except ImportError:
    print("Installing deep_translator...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "deep_translator", "-q"])
    print("deep_translator installed.")

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # go up from notebooks to project root
file_path = PROJECT_ROOT / "data" / "processed" / "02_lyrics_lang.csv"
# DATABRICKS PATH
# file_path = "/Volumes/songs_db/default/storage/02_lyrics_lang.csv"

df = pd.read_csv(file_path)

print(df.head())

                       artist                   title             spotify_uri  \
0  Ryan Castro, Kapo, Gangsta                LA VILLA  2ZyrAym0sRLwt4PhGotHuI   
1                   Bad Bunny       BAILE INoLVIDABLE  2lTm559tuIvatlT1u0JYG2   
2       El Bogueto, Yung Beef  Cuando No Era Cantante  6N2iccqxRInhTLHc2Fu3W0   
3                 La T y La M              Soy Favela  3TfRpsYPQSXqqramSoWlNg   
4                   Max Carra  UWAIE - versión cumbia  6UO9DhCUq4ZbQz7PXwelFV   

                                              lyrics  length original_lang  
0  Kapo, Ryan Castro, Gangsta\nQué chimba, SOG\nT...    2484            es  
1  Pensaba que contigo iba a envejecer\nEn otra v...    2083            es  
2  Jajajajaja\nJaja, QueHicisteBella\n\nComo ante...    2515            es  
3  ¡Ay!\n\nHe caminado tantas calles pero no he e...    1429            es  
4                                                NaN       0       unknown  


In [16]:
import duckdb as dd
con = dd.connect()
lang_groups = con.execute("SELECT original_lang, COUNT(*) AS count FROM df GROUP BY original_lang order by original_lang").df()

display(lang_groups)

,original_lang,count
0,ar,1
1,de,1
2,en,162
3,eo,1
4,es,276
5,fr,1
6,gd,1
7,he,1
8,hr,1
9,id,1


In [17]:
# Cell 4
from pathlib import Path
import re
import pandas as pd
import concurrent.futures as cf
from deep_translator import GoogleTranslator

LANGUAGE_CODE_MAP = {
    "zh-cn": "zh-CN",
    "zh_cn": "zh-CN",
    "zh-tw": "zh-TW",
    "zh_tw": "zh-TW",
    "jp": "ja",
    "kr": "ko",
    "latin": "auto",
    "other": "auto",
    "unknown": "auto",
}

CJK_PATTERN = re.compile(r"[\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff]")
TRANSLATE_TIMEOUT_SEC = 30
translator_cache = {}

def normalize_source_language(language_value):
    if pd.isna(language_value):
        return "auto"
    language = str(language_value).strip().lower()
    if not language:
        return "auto"
    return LANGUAGE_CODE_MAP.get(language, language)

def has_cjk(text):
    if pd.isna(text):
        return False
    return bool(CJK_PATTERN.search(str(text)))

def get_translator(source_language):
    if source_language not in translator_cache:
        translator_cache[source_language] = GoogleTranslator(source=source_language, target="en")
    return translator_cache[source_language]

def translate_once(text, source_language, timeout_sec=TRANSLATE_TIMEOUT_SEC):
    try:
        translator = get_translator(source_language)
        with cf.ThreadPoolExecutor(max_workers=1) as ex:
            fut = ex.submit(translator.translate, text)
            return fut.result(timeout=timeout_sec)
    except cf.TimeoutError:
        return pd.NA
    except Exception:
        return pd.NA

def translate_line_by_line(text, source_language):
    lines = str(text).splitlines()
    out_lines = []

    for line in lines:
        line_stripped = line.strip()
        if not line_stripped:
            out_lines.append("")
            continue

        translated = translate_once(line_stripped, source_language)
        if pd.isna(translated):
            translated = translate_once(line_stripped, "auto")
        if pd.isna(translated):
            translated = line_stripped

        out_lines.append(str(translated))

    return "\n".join(out_lines)

def translate_to_english(text, source_language="auto"):
    if pd.isna(text):
        return pd.NA

    original = str(text).strip()
    if not original:
        return pd.NA

    normalized_source = normalize_source_language(source_language)

    first_try = translate_once(original, normalized_source)
    if pd.isna(first_try) and normalized_source != "auto":
        first_try = translate_once(original, "auto")

    if pd.isna(first_try):
        first_try = translate_line_by_line(original, normalized_source)

    unchanged = (not pd.isna(first_try)) and (str(first_try).strip() == original)
    still_cjk = (not pd.isna(first_try)) and has_cjk(first_try)

    if unchanged or still_cjk:
        retry_auto = translate_once(original, "auto")
        if not pd.isna(retry_auto) and str(retry_auto).strip() != original and not has_cjk(retry_auto):
            return retry_auto

        retry_lines = translate_line_by_line(original, "auto")
        if retry_lines and retry_lines.strip():
            return retry_lines

    return first_try

In [18]:
# Cell 5
 # Translate non-English rows with a length gate and review flags
df_translated = df.copy()

if "lyrics" not in df_translated.columns or "original_lang" not in df_translated.columns:
    raise KeyError("Expected columns 'lyrics' and 'original_lang' in the input CSV")

output_path = PROJECT_ROOT / "data" / "processed" / "03_lyrics_trans.csv"
# DATABRICKS PATH
# output_path = Path("/Volumes/songs_db/default/storage/03_lyrics_trans.csv")
checkpoint_every = 1

TRANSLATE_MAX_LENGTH = 5000  # review_required is True only when lyrics length > 5000

if "length" not in df_translated.columns:
    df_translated["length"] = df_translated["lyrics"].fillna("").astype(str).str.len()
else:
    df_translated["length"] = pd.to_numeric(df_translated["length"], errors="coerce").fillna(0).astype(int)

# Final review flag is set after translation values are ready.
df_translated["translation_review_required"] = False

# Resume support: continue from prior partial output if present
if output_path.exists():
    existing = pd.read_csv(output_path)
    if "lyrics_in_en" in existing.columns and len(existing) == len(df_translated):
        df_translated["lyrics_in_en"] = existing["lyrics_in_en"]
    else:
        df_translated["lyrics_in_en"] = pd.NA
else:
    df_translated["lyrics_in_en"] = pd.NA

non_english_mask = df_translated["original_lang"].fillna("").str.lower() != "en"
english_mask = df_translated["original_lang"].fillna("").str.lower() == "en"
has_lyrics_mask = df_translated["lyrics"].notna()

eligible_length_mask = df_translated["length"] <= TRANSLATE_MAX_LENGTH
too_long_mask = non_english_mask & has_lyrics_mask & (~eligible_length_mask)

existing_en = df_translated["lyrics_in_en"].fillna("").astype(str).str.strip()
original_lyrics = df_translated["lyrics"].fillna("").astype(str).str.strip()

still_cjk_mask = existing_en.str.contains(r"[\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff]", regex=True)
unchanged_mask = existing_en.eq(original_lyrics) & existing_en.ne("")

needs_translation_mask = non_english_mask & has_lyrics_mask & eligible_length_mask & (
    df_translated["lyrics_in_en"].isna() | still_cjk_mask | unchanged_mask
)

pending_indices = df_translated.index[needs_translation_mask].tolist()
total_pending = len(pending_indices)
print(f"Pending translations (length <= {TRANSLATE_MAX_LENGTH}): {total_pending:,}")
print(f"Skipped for manual review due to length > {TRANSLATE_MAX_LENGTH}: {int(too_long_mask.sum()):,}")

since_last_save = 0
processed = 0
failed_rows = []

for idx in pending_indices:
    uri = df_translated.at[idx, "spotify_uri"] if "spotify_uri" in df_translated.columns else idx
    print(f"Translating {processed + 1}/{total_pending} | uri={uri} | lang={df_translated.at[idx, 'original_lang']}")

    translated = translate_to_english(
        df_translated.at[idx, "lyrics"],
        source_language=df_translated.at[idx, "original_lang"],
    )

    if pd.isna(translated):
        failed_rows.append(uri)

    df_translated.at[idx, "lyrics_in_en"] = translated
    since_last_save += 1
    processed += 1

    if since_last_save >= checkpoint_every:
        df_translated[["artist", "title", "spotify_uri", "original_lang", "lyrics_in_en", "length", "translation_review_required"]].to_csv(output_path, index=False)
        print(f"Progress: {processed}/{total_pending} (checkpoint saved)")
        since_last_save = 0

# For English rows, copy lyrics to lyrics_in_en
df_translated.loc[english_mask, "lyrics_in_en"] = df_translated.loc[english_mask, "lyrics"]

# Rule 2: suspicious translated text (non-English scripts present in lyrics_in_en).
suspicious_translation_mask = df_translated["lyrics_in_en"].fillna("").astype(str).str.contains(
    r"[\u0400-\u04FF\u0590-\u05FF\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\u0900-\u097F\u0980-\u09FF\u0A00-\u0A7F\u0A80-\u0AFF\u0B00-\u0B7F\u0B80-\u0BFF\u0C00-\u0C7F\u0C80-\u0CFF\u0D00-\u0D7F\u0E00-\u0E7F\u0E80-\u0EFF\u10A0-\u10FF\u1100-\u11FF\u3040-\u30FF\u3400-\u4DBF\u4E00-\u9FFF\uAC00-\uD7AF\uF900-\uFAFF]",
    regex=True,
    na=False,
 )

# Final review logic (only your two conditions).
df_translated["translation_review_required"] = (df_translated["length"] > TRANSLATE_MAX_LENGTH) | suspicious_translation_mask

# Final save (include review metadata for Phase 4)
df_translated[["artist", "title", "spotify_uri", "original_lang", "lyrics_in_en", "length", "translation_review_required"]].to_csv(output_path, index=False)

print(f"Progress: {processed}/{total_pending} (final save)")
print(f"Wrote {len(df_translated):,} rows to {output_path}")
print(f"Failed rows (timeout/error): {len(failed_rows):,}")

if failed_rows:
    failed_path = output_path.with_name("lyrics_trans_failed_uris.csv")
    pd.DataFrame({"spotify_uri": failed_rows}).to_csv(failed_path, index=False)
    print(f"Wrote failed URI list to {failed_path}")

display(
    df_translated[["spotify_uri", "original_lang", "length", "translation_review_required", "lyrics_in_en"]].head(10)
)

Pending translations (length <= 5000): 480
Skipped for manual review due to length > 5000: 10
Translating 1/480 | uri=2ZyrAym0sRLwt4PhGotHuI | lang=es
Progress: 1/480 (checkpoint saved)
Translating 2/480 | uri=2lTm559tuIvatlT1u0JYG2 | lang=es
Progress: 2/480 (checkpoint saved)
Translating 3/480 | uri=6N2iccqxRInhTLHc2Fu3W0 | lang=es
Progress: 3/480 (checkpoint saved)
Translating 4/480 | uri=3TfRpsYPQSXqqramSoWlNg | lang=es
Progress: 4/480 (checkpoint saved)
Translating 5/480 | uri=4AL4EamHEBKPpdcFRkYdXN | lang=es
Progress: 5/480 (checkpoint saved)
Translating 6/480 | uri=3sK8wGT43QFpWrvNQsrQya | lang=es
Progress: 6/480 (checkpoint saved)
Translating 7/480 | uri=6X8DTIJEgHUjZynuds0E2f | lang=es
Progress: 7/480 (checkpoint saved)
Translating 8/480 | uri=59D4DOkspUbWyMmbAPQkxZ | lang=es
Progress: 8/480 (checkpoint saved)
Translating 9/480 | uri=1YAWMfsmduIrpqrjP45pFn | lang=es
Progress: 9/480 (checkpoint saved)
Translating 10/480 | uri=6uiQ0DqbXTBc8tl0AUKZkR | lang=es
Progress: 10/480 (ch

,spotify_uri,original_lang,length,translation_review_required,lyrics_in_en
0,2ZyrAym0sRLwt4PhGotHuI,es,2484,False,"Kapo, Ryan Castro, Gangsta\nWhat a joke, SOG\n..."
1,2lTm559tuIvatlT1u0JYG2,es,2083,False,I thought I was going to grow old with you\nIn...
2,6N2iccqxRInhTLHc2Fu3W0,es,2515,False,"Hahahahaha\nHaha, What Did You DoBella\n\nLike..."
3,3TfRpsYPQSXqqramSoWlNg,es,1429,False,Oh!\n\nI have walked so many streets but I hav...
4,6UO9DhCUq4ZbQz7PXwelFV,unknown,0,False,<NA>
5,4AL4EamHEBKPpdcFRkYdXN,es,1350,False,"If your mirror could talk, it would tell you t..."
6,3sK8wGT43QFpWrvNQsrQya,es,2403,False,"Hey, hey, hey, hey\n\nAnother beautiful sunset..."
7,6X8DTIJEgHUjZynuds0E2f,es,1576,False,"With this topic, I'm going to make you travel ..."
8,59D4DOkspUbWyMmbAPQkxZ,es,1923,False,"Acho, PR is something else\nI met her in Miami..."
9,2pbSCYzxrG0wa6qcj8IyiE,en,5006,True,Uh-uh-uh\nHow pretty that ass loo-oo-ooks\nIn ...
